# LSTM from Scratch (NumPy)

In this notebook, we build and train an LSTM **from first principles** using only NumPy.
We will:
1. Prepare a tiny character dataset.
2. Implement LSTM equations manually.
3. Derive and code backpropagation through time (BPTT).
4. Train with gradient clipping + AdaGrad.
5. Sample generated text from the trained model.

Each step includes an explanation so you can connect the math to code.

## Step 1 — Imports and Reproducibility

We only use NumPy for tensor math and Matplotlib for a loss curve.
A fixed random seed ensures consistent initialization and repeatable results.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## Step 2 — Build a Small Character-Level Dataset

We train a next-character language model.
Given a character at time *t*, the model predicts the next character at *t+1*.

Using a small corpus keeps training fast, while still demonstrating full LSTM mechanics.

In [ ]:
text = (
    "lstm from scratch is powerful. "
    "lstm learns long term dependencies. "
    "recurrent networks remember context. "
) * 20

chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}

print(f"Corpus length: {len(text)}")
print(f"Vocab size: {vocab_size}")
print("Vocabulary:", ''.join(chars))

## Step 3 — One-Hot Encoding Helpers

At each time step, input x_t is a one-hot vector of length `vocab_size`.
If the current character index is `k`, then `x_t[k] = 1` and all others are `0`.

This representation is simple and makes matrix multiplications easy to follow.

In [ ]:
def one_hot(idx, size):
    v = np.zeros((size, 1))
    v[idx] = 1.0
    return v

## Step 4 — Activation Functions

LSTM uses:
- Sigmoid for gates: values in [0, 1], controlling information flow.
- Tanh for candidate/update vectors: values in [-1, 1].
- Softmax for output probabilities over vocabulary.

We also include derivatives for backpropagation.

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def dsigmoid(y):
    return y * (1.0 - y)

def dtanh(y):
    return 1.0 - y**2

def softmax(x):
    z = x - np.max(x)
    e = np.exp(z)
    return e / np.sum(e)

## Step 5 — Initialize LSTM Parameters

For input x_t and previous hidden state h_(t-1), we concatenate z_t = [h_(t-1); x_t].
Then compute forget/input/output gates and candidate memory, update cell state c_t, then hidden state h_t.
Finally, produce logits and softmax probabilities for the next character.

In [ ]:
hidden_size = 64
seq_length = 25
learning_rate = 0.08

concat_size = hidden_size + vocab_size
scale = 1.0 / np.sqrt(concat_size)

params = {
    'Wf': np.random.randn(hidden_size, concat_size) * scale,
    'Wi': np.random.randn(hidden_size, concat_size) * scale,
    'Wo': np.random.randn(hidden_size, concat_size) * scale,
    'Wg': np.random.randn(hidden_size, concat_size) * scale,
    'bf': np.zeros((hidden_size, 1)),
    'bi': np.zeros((hidden_size, 1)),
    'bo': np.zeros((hidden_size, 1)),
    'bg': np.zeros((hidden_size, 1)),
    'Wy': np.random.randn(vocab_size, hidden_size) * (1.0 / np.sqrt(hidden_size)),
    'by': np.zeros((vocab_size, 1)),
}

mem = {k: np.zeros_like(v) for k, v in params.items()}

## Step 6 — Forward Pass for One Sequence

We unroll the LSTM across `seq_length` time steps.
At every step, we save intermediate tensors so backward pass can reuse them.
Loss is summed cross-entropy over all time steps.

In [ ]:
def forward(inputs, targets, h_prev, c_prev, params):
    xs, zs = {}, {}
    fs, is_, os, gs = {}, {}, {}, {}
    cs, hs, ys, ps = {}, {}, {}, {}

    hs[-1] = h_prev.copy()
    cs[-1] = c_prev.copy()

    loss = 0.0

    for t in range(len(inputs)):
        xs[t] = one_hot(inputs[t], vocab_size)
        zs[t] = np.vstack((hs[t - 1], xs[t]))

        fs[t] = sigmoid(params['Wf'] @ zs[t] + params['bf'])
        is_[t] = sigmoid(params['Wi'] @ zs[t] + params['bi'])
        os[t] = sigmoid(params['Wo'] @ zs[t] + params['bo'])
        gs[t] = np.tanh(params['Wg'] @ zs[t] + params['bg'])

        cs[t] = fs[t] * cs[t - 1] + is_[t] * gs[t]
        hs[t] = os[t] * np.tanh(cs[t])

        ys[t] = params['Wy'] @ hs[t] + params['by']
        ps[t] = softmax(ys[t])

        loss += -np.log(ps[t][targets[t], 0] + 1e-12)

    cache = (xs, zs, fs, is_, os, gs, cs, hs, ys, ps)
    return loss, cache, hs[len(inputs) - 1], cs[len(inputs) - 1]

## Step 7 — Backpropagation Through Time (BPTT)

We walk backward from last time step to first.
Key ideas:
- Output gradient uses (p_t - one_hot(target)).
- Hidden gradient combines output path and recurrent path from next time step.
- Cell gradient combines hidden-to-cell path and recurrent cell path.

Gradient clipping keeps updates numerically stable.

In [ ]:
def backward(inputs, targets, cache, params, h_last_grad=None, c_last_grad=None):
    xs, zs, fs, is_, os, gs, cs, hs, ys, ps = cache
    T = len(inputs)

    grads = {k: np.zeros_like(v) for k, v in params.items()}

    dh_next = np.zeros((hidden_size, 1)) if h_last_grad is None else h_last_grad
    dc_next = np.zeros((hidden_size, 1)) if c_last_grad is None else c_last_grad

    for t in reversed(range(T)):
        dy = ps[t].copy()
        dy[targets[t]] -= 1.0

        grads['Wy'] += dy @ hs[t].T
        grads['by'] += dy

        dh = params['Wy'].T @ dy + dh_next

        tanh_c = np.tanh(cs[t])
        do = dh * tanh_c
        do_raw = do * dsigmoid(os[t])

        dc = dh * os[t] * dtanh(tanh_c) + dc_next

        df = dc * cs[t - 1]
        df_raw = df * dsigmoid(fs[t])

        di = dc * gs[t]
        di_raw = di * dsigmoid(is_[t])

        dg = dc * is_[t]
        dg_raw = dg * dtanh(gs[t])

        grads['Wf'] += df_raw @ zs[t].T
        grads['bf'] += df_raw
        grads['Wi'] += di_raw @ zs[t].T
        grads['bi'] += di_raw
        grads['Wo'] += do_raw @ zs[t].T
        grads['bo'] += do_raw
        grads['Wg'] += dg_raw @ zs[t].T
        grads['bg'] += dg_raw

        dz = (
            params['Wf'].T @ df_raw
            + params['Wi'].T @ di_raw
            + params['Wo'].T @ do_raw
            + params['Wg'].T @ dg_raw
        )

        dh_next = dz[:hidden_size, :]
        dc_next = dc * fs[t]

    for k in grads:
        np.clip(grads[k], -5, 5, out=grads[k])

    return grads

## Step 8 — Parameter Update (AdaGrad)

AdaGrad scales each parameter's step size using historical squared gradients.
This usually improves stability for RNN/LSTM training without heavy tuning.

In [ ]:
def adagrad_step(params, grads, mem, lr=0.08, eps=1e-8):
    for k in params:
        mem[k] += grads[k] * grads[k]
        params[k] -= lr * grads[k] / (np.sqrt(mem[k]) + eps)

## Step 9 — Text Sampling Function

We generate text one character at a time by feeding the previous sampled character back into the model.

In [ ]:
def sample(seed_idx, n, h, c, params):
    x = one_hot(seed_idx, vocab_size)
    out_indices = []

    for _ in range(n):
        z = np.vstack((h, x))
        f = sigmoid(params['Wf'] @ z + params['bf'])
        i = sigmoid(params['Wi'] @ z + params['bi'])
        o = sigmoid(params['Wo'] @ z + params['bo'])
        g = np.tanh(params['Wg'] @ z + params['bg'])

        c = f * c + i * g
        h = o * np.tanh(c)

        y = params['Wy'] @ h + params['by']
        p = softmax(y)
        idx = np.random.choice(range(vocab_size), p=p.ravel())

        x = one_hot(idx, vocab_size)
        out_indices.append(idx)

    return ''.join(idx_to_char[i] for i in out_indices)

## Step 10 — Training Loop

For each iteration:
1. Slice a chunk of text.
2. Build input and shifted target sequences.
3. Run forward + backward.
4. Apply AdaGrad update.
5. Track smoothed loss and print samples occasionally.

In [ ]:
data = [char_to_idx[ch] for ch in text]

h_prev = np.zeros((hidden_size, 1))
c_prev = np.zeros((hidden_size, 1))

smooth_loss = -np.log(1.0 / vocab_size) * seq_length
loss_history = []

pointer = 0
iterations = 1200

for it in range(iterations):
    if pointer + seq_length + 1 >= len(data):
        h_prev = np.zeros((hidden_size, 1))
        c_prev = np.zeros((hidden_size, 1))
        pointer = 0

    inputs = data[pointer:pointer + seq_length]
    targets = data[pointer + 1:pointer + seq_length + 1]

    loss, cache, h_prev, c_prev = forward(inputs, targets, h_prev, c_prev, params)
    grads = backward(inputs, targets, cache, params)
    adagrad_step(params, grads, mem, lr=learning_rate)

    smooth_loss = smooth_loss * 0.99 + loss * 0.01
    loss_history.append(smooth_loss)

    if it % 200 == 0:
        seed = inputs[0]
        snippet = sample(seed, 120, h_prev.copy(), c_prev.copy(), params)
        print(f"Iter {it:4d} | smooth loss {smooth_loss:.3f}")
        print(snippet)
        print("-" * 60)

    pointer += seq_length

## Step 11 — Visualize Loss

A downward trend confirms that the LSTM is learning the sequence distribution.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(loss_history)
plt.title('Smoothed training loss')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

## Step 12 — Final Sample

After training, sample from an all-zero initial hidden/cell state.

In [ ]:
seed_char = 'l'
print('Seed:', seed_char)
print(sample(char_to_idx[seed_char], 220, np.zeros((hidden_size, 1)), np.zeros((hidden_size, 1)), params))

## What You Built

You completed a full LSTM from scratch pipeline:
- manual gate equations,
- manual BPTT through time,
- clipped gradients,
- AdaGrad optimizer,
- autoregressive text generation.

This is the conceptual core behind framework-based LSTM implementations.